# **02_Data_Cleaning.py**




Goal:
The raw Elliptic and Ethereum datasets should be cleaned and ready for machine learning modeling and exploratory analysis.

Inputs:
The Raw_data folder contains the raw datasets.


*   elliptic_txs_features.csv   
*   elliptic_txs_classes.csv
*    transaction_dataset.csv






Processing
The original raw files are preserved while the data is examined and cleansed.
Prior to modeling, issues with missing values, duplicate entries, column formats, and dataset-specific constraints are resolved.

Results:
datasets that have been cleaned and stored in the cleaned_data folder for use in later rounds of modeling and exploratory research.

In [1]:
import os
import pandas as pd

In [2]:
raw_data_path = "../Raw_data"
cleaned_data_path = "../cleaned_data"
os.makedirs(cleaned_data_path, exist_ok=True)

print("Files in Raw_data:")

for file in os.listdir(raw_data_path):
    print(file)

Files in Raw_data:
elliptic_txs_classes.csv
elliptic_txs_edgelist.csv
elliptic_txs_features.csv
transaction_dataset.csv


In [3]:
# ELLIPTIC

feats = pd.read_csv("../Raw_data/elliptic_txs_features.csv", header=None)
# 167 columns total: txId + 166 features (feature 1 = time step, per Weber et al. 2019)
feat_cols = ["txId"] + [f"feat_{i}" for i in range(1, 167)]
feats.columns = feat_cols

In [4]:
classes = pd.read_csv("../Raw_data/elliptic_txs_classes.csv")
merged = feats.merge(classes, on="txId", how="left")

In [5]:
# Drop unlabelled rows - not usable for supervised classification
merged_labelled = merged[merged["class"] != "unknown"].copy()
merged_labelled["label"] = merged_labelled["class"].map({"1": 1, "2": 0})  # 1=illicit, 0=licit
merged_labelled = merged_labelled.drop(columns=["class"])

In [6]:
print(f"Elliptic: {merged_labelled.shape[0]} labelled rows "
      f"({merged_labelled['label'].sum()} illicit, "
      f"{(merged_labelled['label']==0).sum()} licit)")
merged_labelled.to_csv(os.path.join(cleaned_data_path, "elliptic_clean.csv"), index=False)

Elliptic: 46564 labelled rows (4545 illicit, 42019 licit)


In [7]:
# ETHEREUM

eth = pd.read_csv("../Raw_data/transaction_dataset.csv")
eth = eth.drop(columns=[c for c in ["Unnamed: 0", "Index"] if c in eth.columns])


In [8]:
# De-duplicate on Address before further processing or splitting.
# ID/index columns are excluded because they can mask duplicate records.

before = len(eth)
eth = eth.drop_duplicates(subset="Address", keep="first").copy()
print(f"Ethereum: removed {before - len(eth)} duplicate-address rows")

Ethereum: removed 25 duplicate-address rows


In [9]:
erc20_categorical = [" ERC20 most sent token type", " ERC20_most_rec_token_type"]
erc20_numeric = [c for c in eth.columns if "ERC20" in c and c not in erc20_categorical]

In [10]:
eth["has_erc20_activity"] = eth[erc20_numeric].notna().any(axis=1).astype(int)
eth[erc20_numeric] = eth[erc20_numeric].fillna(0)
eth = eth.drop(columns=erc20_categorical)

In [11]:
print(f"Ethereum: {eth.shape[0]} rows, {eth['FLAG'].sum()} fraudulent "
      f"({eth['FLAG'].mean():.1%}), missing values remaining: {eth.isna().sum().sum()}")
eth.to_csv(os.path.join(cleaned_data_path, "ethereum_clean.csv"), index=False)

Ethereum: 9816 rows, 2179 fraudulent (22.2%), missing values remaining: 0


In [12]:
project_path = ".."

cleaned_data_path = os.path.join(project_path, "cleaned_data")

os.makedirs(cleaned_data_path, exist_ok=True)

print("Cleaned data folder:", cleaned_data_path)

Cleaned data folder: ../cleaned_data


In [13]:
elliptic_output = os.path.join(cleaned_data_path, "elliptic_clean.csv")
ethereum_output = os.path.join(cleaned_data_path, "ethereum_clean.csv")

print("Cleaned datasets saved successfully.")
print(elliptic_output)
print(ethereum_output)

Cleaned datasets saved successfully.
../cleaned_data/elliptic_clean.csv
../cleaned_data/ethereum_clean.csv


In [14]:
print("Cleaned files:")
for file in os.listdir(cleaned_data_path):
    print(file)

Cleaned files:
elliptic_clean.csv
ethereum_clean.csv
